## 👨‍💼 Business Scenario

You are working as a Data Analyst at an e-commerce company.

The Operations Manager proposes:

> "We are considering investing in faster delivery. Before spending more money, I want to know whether faster delivery could improve customer satisfaction."
> 

Your task is to use historical order data to investigate the relationship and then design a proper A/B experiment.

In [1]:
import pandas as pd
import seaborn as sns

In [2]:
cd = pd.read_csv("olist_customers_dataset.csv") 
gd=pd.read_csv("olist_geolocation_dataset.csv")
ot=pd.read_csv("olist_order_items_dataset.csv")
od=pd.read_csv("olist_orders_dataset.csv")
pyd=pd.read_csv("olist_order_payments_dataset.csv")
ord=pd.read_csv("olist_order_reviews_dataset.csv")
odd=pd.read_csv("olist_orders_dataset.csv")
prd=pd.read_csv("olist_products_dataset.csv")
osd=pd.read_csv("olist_sellers_dataset.csv")
pcn=pd.read_csv("product_category_name_translation.csv")

In [3]:
od

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00
...,...,...,...,...,...,...,...,...
99436,9c5dedf39a927c1b2549525ed64a053c,39bd1228ee8140590ac3aca26f2dfe00,delivered,2017-03-09 09:54:05,2017-03-09 09:54:05,2017-03-10 11:18:03,2017-03-17 15:08:01,2017-03-28 00:00:00
99437,63943bddc261676b46f01ca7ac2f7bd8,1fca14ff2861355f6e5f14306ff977a7,delivered,2018-02-06 12:58:58,2018-02-06 13:10:37,2018-02-07 23:22:42,2018-02-28 17:37:56,2018-03-02 00:00:00
99438,83c1379a015df1e13d02aae0204711ab,1aa71eb042121263aafbe80c1b562c9c,delivered,2017-08-27 14:46:43,2017-08-27 15:04:16,2017-08-28 20:52:26,2017-09-21 11:24:17,2017-09-27 00:00:00
99439,11c177c8e97725db2631073c19f07b62,b331b74b18dc79bcdf6532d51e1637c1,delivered,2018-01-08 21:28:27,2018-01-08 21:36:21,2018-01-12 15:35:03,2018-01-25 23:32:54,2018-02-15 00:00:00


In [4]:
od['order_delivered_customer_date']=pd.to_datetime(od['order_delivered_customer_date'])
od['order_purchase_timestamp']=pd.to_datetime(od['order_purchase_timestamp'])

In [5]:
od['actual_delivery_time']=(od['order_delivered_customer_date']-od['order_purchase_timestamp']).dt.total_seconds() / (24 * 60 * 60)

In [6]:
ord.describe()

,review_score
count,99224.000000
mean,4.086421
std,1.347579
min,1.000000
25%,4.000000
50%,5.000000
75%,5.000000
max,5.000000


In [7]:
data_relation=od[['order_id','actual_delivery_time']].merge(ord[['order_id','review_score']],on='order_id',how='left')

In [8]:
data_relation['actual_delivery_time']=(data_relation['actual_delivery_time'].round(0))

In [9]:
data_relation

,order_id,actual_delivery_time,review_score
0,e481f51cbdc54678b7cc49136f2d6af7,8.0,4.0
1,53cdb2fc8bc7dce0b6741e2150273451,14.0,4.0
2,47770eb9100c2d0c44946d9cf07ec65d,9.0,5.0
3,949d5b44dbf5de918fe9c16f97b45f8a,13.0,5.0
4,ad21c59c0840e6cb83a9ceb5573f8159,3.0,5.0
...,...,...,...
99987,9c5dedf39a927c1b2549525ed64a053c,8.0,5.0
99988,63943bddc261676b46f01ca7ac2f7bd8,22.0,4.0
99989,83c1379a015df1e13d02aae0204711ab,25.0,5.0
99990,11c177c8e97725db2631073c19f07b62,17.0,2.0


In [10]:
data_relation.describe()

,actual_delivery_time,review_score
count,97005.000000,99224.000000
mean,12.523385,4.086421
std,9.546804,1.347579
min,1.000000,1.000000
25%,7.000000,4.000000
50%,10.000000,5.000000
75%,16.000000,5.000000
max,210.000000,5.000000


In [11]:
arv=data_relation.groupby('actual_delivery_time')['review_score'].mean().reset_index()

In [12]:
data_relation.describe()

,actual_delivery_time,review_score
count,97005.000000,99224.000000
mean,12.523385,4.086421
std,9.546804,1.347579
min,1.000000,1.000000
25%,7.000000,4.000000
50%,10.000000,5.000000
75%,16.000000,5.000000
max,210.000000,5.000000


In [13]:
data_relation.duplicated().sum()

np.int64(349)

In [14]:
data_relation.drop_duplicates(inplace=True)

In [15]:
data_relation.isnull().sum()

order_id                   0
actual_delivery_time    2978
review_score             768
dtype: int64

In [16]:
data_relation.dropna(inplace=True)

In [18]:
data_relation.describe()

,actual_delivery_time,review_score
count,96019.000000,96019.000000
mean,12.480759,4.154553
std,9.465961,1.285491
min,1.000000,1.000000
25%,7.000000,4.000000
50%,10.000000,5.000000
75%,16.000000,5.000000
max,208.000000,5.000000


In [19]:
data_relation

,order_id,actual_delivery_time,review_score
0,e481f51cbdc54678b7cc49136f2d6af7,8.0,4.0
1,53cdb2fc8bc7dce0b6741e2150273451,14.0,4.0
2,47770eb9100c2d0c44946d9cf07ec65d,9.0,5.0
3,949d5b44dbf5de918fe9c16f97b45f8a,13.0,5.0
4,ad21c59c0840e6cb83a9ceb5573f8159,3.0,5.0
...,...,...,...
99987,9c5dedf39a927c1b2549525ed64a053c,8.0,5.0
99988,63943bddc261676b46f01ca7ac2f7bd8,22.0,4.0
99989,83c1379a015df1e13d02aae0204711ab,25.0,5.0
99990,11c177c8e97725db2631073c19f07b62,17.0,2.0


In [23]:
data_relation.groupby('review_score')['actual_delivery_time'].mean().reset_index()

,review_score,actual_delivery_time
0,1.0,21.264066
1,2.0,16.609939
2,3.0,14.224097
3,4.0,12.268490
4,5.0,10.643432


In [26]:
data_relation['actual_delivery_time'].median()

np.float64(10.0)

### Group A — Slower Delivery

Orders with delivery time greater than the median delivery time.

### Group B — Faster Delivery

Orders with delivery time less than or equal to the median delivery time.

In [27]:
group_A=data_relation[data_relation['actual_delivery_time']>data_relation['actual_delivery_time'].median()]

In [28]:
group_B=data_relation[data_relation['actual_delivery_time']<=data_relation['actual_delivery_time'].median()]

In [ ]:
#historical comparison groups
median_delivery = data_relation["actual_delivery_time"].median()
print("Median:", median_delivery)
print("Group A:", len(group_A))
print("Group B:", len(group_B))

Median: 10.0
Group A: 46411
Group B: 49608


In [31]:
print(group_A["actual_delivery_time"].min())
print(group_B["actual_delivery_time"].max())

11.0
10.0
